# 랭체인(LangChain) SQL Query validation 예제
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## Reference : https://python.langchain.com/docs/use_cases/sql/query_checking

![](https://python.langchain.com/assets/images/sql_usecase-d432701261f05ab69b38576093718cf3.png)

# Sample SQL DB 다운로드

## Reference : https://www.sqlitetutorial.net/sqlite-sample-database/

![](https://www.sqlitetutorial.net/wp-content/uploads/2015/11/sqlite-sample-database-color.jpg)

In [ ]:
!wget https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip -O chinook.zip

In [ ]:
!unzip chinook.zip

# LangChain 라이브러리 설치

In [ ]:
!pip install -q langchain langchain-community langchain-openai langchain-core

# OpenAI API Key 설정

In [ ]:
OPENAI_KEY = "input Your Key"

# chinook.db 불러오기

In [ ]:
import os
from langchain_community.utilities import SQLDatabase

db_filename = "chinook.db"


db = SQLDatabase.from_uri(f"sqlite:///{db_filename}")

# 4. 연결 테스트 (employees 테이블이 보여야 정상)
print(f"테이블 목록: {db.get_table_names()}")

# Query checker

* 가장 간단한 전략은 모델에게 원본 쿼리에서 흔한 실수를 확인하도록 요청하는 것입니다. 다음과 같은 SQL 쿼리 체인을 가정해 보겠습니다.







In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# 1. LLM 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=OPENAI_KEY)


# 2-1. 프롬프트 정의
template = """You are a SQLite expert. Given an input question, create a syntactically correct SQLite query to run.
Unless the user specifies otherwise, limit your results to at most 5.
Never query for all columns from a table. You must query only the columns that are needed to answer the question.
Pay attention to use only the column names you can see in the tables below.

[Schema]
{schema}

[Question]
{question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 스키마 조회 함수
def get_schema(_):
    return db.get_table_info()

# 체인 조립
chain = (
    RunnablePassthrough.assign(schema=get_schema) # DB에서 스키마 가져오기
    | prompt                                      # 프롬프트에 넣기
    | llm                                         # GPT-4에게 질문
    | StrOutputParser()                           # 결과를 문자열로 변환
)

* 그리고 우리는 그 출력을 검증하고 싶습니다. 체인을 두 번째 프롬프트와 모델 호출로 확장함으로써 그렇게 할 수 있습니다:

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

system = """사용자의 {dialect} 쿼리를 다음과 같은 흔한 실수에 대해 다시 확인하세요:
- NULL 값에 NOT IN을 사용하는 경우
- UNION을 사용할 때 UNION ALL을 사용해야 하는 경우
- 배타적 범위에 BETWEEN을 사용하는 경우
- 조건문에서 데이터 타입 불일치
- 식별자를 올바르게 인용하는 경우
- 함수에 올바른 인수 수를 사용하는 경우
- 올바른 데이터 타입으로 캐스팅하는 경우
- 조인에 적합한 컬럼을 사용하는 경우

위의 실수 중 어느 것이라도 있다면, 쿼리를 다시 작성하세요. 실수가 없다면, 원본 쿼리를 그대로 재생산하세요.

최종 SQL 쿼리만 출력하세요."""
prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("human", "{query}")]
).partial(dialect=db.dialect)
validation_chain = prompt | llm | StrOutputParser()

full_chain = {"query": chain} | validation_chain

In [ ]:
query = full_chain.invoke(
    {
        "question": "What's the average Invoice from an American customer whose Fax is missing since 2003 but before 2010"
    }
)
query

In [ ]:
clean_query = query.replace("```sql", "").replace("```", "").strip() #마크다운 삭제
result = db.run(clean_query)
result

* 이 접근법의 명백한 단점은 쿼리를 생성하기 위해 하나 대신 두 번의 모델 호출을 해야 한다는 것입니다. 이를 해결하기 위해 우리는 쿼리 생성과 쿼리 검사를 단일 모델 호출에서 수행하려고 시도할 수 있습니다:

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. 프롬프트 템플릿 정의 (기존 내용 유지)
system_template = """당신은 {dialect} 전문가입니다. 주어진 입력 질문에 대해 문법적으로 정확한 {dialect} 쿼리를 작성하십시오.
사용자가 질문에서 특정한 예시의 수를 지정하지 않은 경우, {dialect}에 따라 LIMIT 절을 사용하여 최대 {top_k}개의 결과를 조회하십시오.
테이블의 모든 컬럼을 조회해서는 안 됩니다. 질문에 답하기 위해 필요한 컬럼만 조회해야 합니다. 각 컬럼 이름을 이중 인용부호(")로 감싸 구분된 식별자로 표시하십시오.
아래 테이블에서 볼 수 있는 컬럼 이름만 사용해야 합니다.

[Schema]
{table_info}

쿼리의 초안을 작성하세요. 그런 다음, {dialect} 쿼리에서 흔한 실수를 다시 확인하고 수정하세요.
아래 형식을 엄격히 준수하세요:

First draft: <<FIRST_DRAFT_QUERY>>
Final answer: <<FINAL_ANSWER_QUERY>>
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_template), ("human", "{input}")]
)


def parse_final_answer(output: str) -> str:
    try:
        # "Final answer:" 기준으로 자르고 뒷부분 가져오기
        text = output.split("Final answer:")[-1].strip()
        # 혹시 모를 마크다운 제거
        return text.replace("```sql", "").replace("```", "").strip()
    except IndexError:
        # 형식을 안 지켰을 경우 전체 반환 (안전장치)
        return output


def get_schema(_):
    return db.get_table_info()

def get_dialect(_):
    return db.dialect


chain = (
    RunnablePassthrough.assign(
        table_info=get_schema,
        dialect=get_dialect,
        top_k=lambda x: 5  # 기본값 5로 설정
    )
    | prompt
    | llm
    | StrOutputParser()
    | parse_final_answer  # 여기서 최종 SQL만 딱 잘라냄
)

# 5. 프롬프트 확인 (요청하신 pretty_print)
print("[프롬프트 구조 확인]")
prompt.pretty_print()



In [ ]:
query = chain.invoke(
    {
        "input": "What's the average Invoice from an USA customer whose Fax is missing since 2003 but before 2010"
    }
)

print(f"생성된 SQL:\n{query}")

In [ ]:
db.run(query)